# 2-13 상장사 문서 검색 — 뉴스·리포트·공시/IR·특허에서 KRX 상장사가 나온 문서만 추리기

**질문 예시:** "이번 주 경제뉴스 중 수주·계약 기사에서 상장사만 뽑아 시장별로 세어줘",
"배터리 특허를 낸 회사 중 상장사는 어디야?"

뉴스 스크랩 웹앱의 핵심 기능입니다. 키워드·카테고리로 `DocumentSearch`를 하고, 문서에 붙은 회사 정보를 **KRX 상장사 목록**과 맞춰
상장사(코스피·코스닥·코넥스, ETF 제외)가 나온 문서만 남긴 뒤 시장별·종목별로 집계합니다.

문서에는 회사 정보가 두 군데 있습니다.
- `securities` — 문서의 **관련 종목**(주제 회사). 뉴스·리포트·공시는 6자리 종목코드, **특허는 출원인 회사 이름**이 들어 있습니다.
- `named_entities` — 본문에서 인식한 **언급 회사**(비상장 포함). NICE 코드·사업자번호·법인번호가 붙어 있습니다.

두 필드를 모두 보고, 코드뿐 아니라 NICE 코드·사업자번호·법인번호로도 상장사를 찾습니다.
끝으로 받은 문서를 **이슈 카테고리**(계약/수주·실적 등 키워드 묶음)와 **관심종목**으로 한 번 더 거릅니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/2_disclosure_news/13_listed_company_docs.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"
D_FROM, D_TO = "2026-11-20", ASOF      # 검색 기간 (DocumentSearch 는 1년 이내)

# 검색 프리셋: 이슈 유형별 키워드 묶음(검색 쿼리와 검색 후 필터에 같이 씀), 언론사 묶음, 관심종목
_A = lambda ws, xs: " or ".join(f"({w} and {x})" for w in ws for x in xs)
ISSUE_PRESETS = {
    "계약/수주": "(수주 and 체결) or (수주 and 공급) or (계약 and 체결) or (계약 and 공급)",
    "인수/투자": "인수 or 합병 or 분할 or 영업양도 or 영업양수 or 엠앤에이 or 출자 or 투자",
    "실적": _A(["매출", "실적", "이익", "배당"], ["발표", "공표", "결정", "기록", "달성", "공시"]),
    "증자/감자": "증자 or 감자 or 주식교환 or 주식이전 or 우회상장",
    "회계/감사": ("상장폐지 or 관리종목 or 자본잠식 or (비적정 and 감사) or (비적정 and 회계법인) or (의견거절 and 감사) "
              "or (의견거절 and 회계법인) or (회계처리 and 위반) or 분식"),
    "소송/부도/회생": "소송 or 횡령 or 배임 or 부도 or 파산 or 회생 or " + _A(["공소", "기소", "혐의"], ["대표이사", "임원", "이사"]),
}
WATCHLIST = ["005930", "000660", "035720", "251970"]   # 관심종목 서식(종목코드 한 열)의 예시 값
PUBLISHER_GROUPS = {
    "중앙일간지": "publisher.raw:('경향신문' or '국민일보' or '동아일보' or '서울신문' or '세계일보' or '아시아투데이' or '조선일보' or '중앙일보' or '한겨레' or '한국일보')",
    "중앙경제지": "publisher.raw:('뉴스토마토' or '디지털타임스' or '매일경제' or '머니투데이' or '서울경제' or '아주경제' or '이데일리' or '이투데이' or '전자신문' or '파이낸셜뉴스' or '한국경제')",
}

## 1. 상장사 목록과 식별자
`GetStockSymbols()` 한 번으로 상장사 전부의 종목코드·NICE 코드·사업자번호·법인번호를 받습니다.
(일반법인 `1`과 공기업 `2`만 — ETF·ETN 등 `8` 제외)

In [4]:
sym = ds_df("GetStockSymbols()")
listed = sym[sym.company_type_l1.isin(["1", "2"]) & sym.market.isin(["KOSPI", "KOSDAQ", "KONEX"])].copy()
listed["code"] = listed.symbol.str.replace("KRX:", "", regex=False)
INFO = listed.set_index("code")[["entity_name", "market"]].to_dict("index")
# 식별번호는 보통주와 우선주가 같은 회사라 겹친다 → 회사 식별용 지도는 보통주(코드 끝자리 0)로만 만든다
common = listed[listed.code.str.endswith("0")]
BY_NICE = dict(zip(common.symbol_nice.dropna(), common.code))
BY_BRID = dict(zip(common.business_rid.dropna().str.replace("-", ""), common.code))
BY_CRID = dict(zip(common.company_rid.dropna().str.replace("-", ""), common.code))
BY_NAME = dict(zip(common.entity_name, common.code))
print(f"상장사 {len(listed):,}개 — NICE {len(BY_NICE):,} · 사업자번호 {len(BY_BRID):,} · 법인번호 {len(BY_CRID):,}")
print(listed.market.value_counts().to_dict())

상장사 2,879개 — NICE 2,770 · 사업자번호 2,749 · 법인번호 2,748
{'KOSDAQ': 1837, 'KOSPI': 935, 'KONEX': 107}


## 2. 검색과 매칭 함수

In [5]:
def fetch_docs(category, sections, query, date_from=D_FROM, date_to=D_TO, pages=1):
    """DocumentSearch 원문(회사 필드 포함) → (총건수, 문서 목록). 한 페이지 100건."""
    docs, total = [], 0
    for p in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(sections)},{json.dumps(query, ensure_ascii=False)},'
             f'count=100,page={p},date_from={date_from},date_to={date_to})')
        c = ds(q)["data"]["pods"][1]["content"]
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        batch = d.get("docs") or []
        docs += batch
        if not batch or p * 100 >= total:   # 페이지가 100건보다 짧게 와도 뒤에 더 있을 수 있음
            break
    return total, docs


def to_code(e):
    """회사 항목 하나 → 상장 종목코드(6자리) 또는 None. 코드 → NICE → 사업자번호 → 법인번호 → 이름 순."""
    s = str(e.get("symbol") or "")
    code = s.split(":")[-1]
    if code in INFO:
        return code
    if s in BY_NICE:
        return BY_NICE[s]
    for key, table in (("business_rid", BY_BRID), ("company_rid", BY_CRID)):
        v = str(e.get(key) or "").replace("-", "")
        if v in table:
            return table[v]
    return BY_NAME.get(e.get("name"))


def match_listed(category, sections, query, date_from=D_FROM, date_to=D_TO, pages=1):
    total, docs = fetch_docs(category, sections, query, date_from, date_to, pages)
    rows = []
    for x in docs:
        rel = {c for c in (to_code(e) for e in (x.get("securities") or [])) if c}
        men = {c for c in (to_code(e) for e in (x.get("named_entities") or []) if isinstance(e, dict)) if c} - rel
        rows.append({"created_at": str(x.get("created_at"))[:16], "publisher": x.get("publisher"),
                     "title": " ".join(str(x.get("title") or "").split())[:70],
                     "관련종목": [INFO[c]["entity_name"] for c in sorted(rel)],
                     "언급종목": [INFO[c]["entity_name"] for c in sorted(men)],
                     "_rel": sorted(rel), "_men": sorted(men), "url": x.get("content_url")})
    df = pd.DataFrame(rows, columns=["created_at", "publisher", "title", "관련종목", "언급종목", "_rel", "_men", "url"])
    n_rel = int((df._rel.str.len() > 0).sum()) if len(df) else 0
    n_any = int(((df._rel.str.len() > 0) | (df._men.str.len() > 0)).sum()) if len(df) else 0
    print(f"[{category} {sections}] '{query[:50]}' 총 {total:,}건 중 {len(df)}건 받음 → "
          f"관련종목에 상장사 {n_rel}건, 언급까지 포함하면 {n_any}건 (+{n_any - n_rel})")
    return df[(df._rel.str.len() > 0) | (df._men.str.len() > 0)].reset_index(drop=True)


def by_stock(df, include_mentions=True):
    """문서 → 종목별 건수(관련/언급)와 시장."""
    rel = pd.Series([c for cs in df._rel for c in cs]).value_counts().rename("관련 문서")
    men = pd.Series([c for cs in df._men for c in cs]).value_counts().rename("언급만 된 문서")
    t = pd.concat([rel, men], axis=1).fillna(0).astype(int)
    if not include_mentions:
        t = t[t["관련 문서"] > 0]
    t["종목"] = [INFO[c]["entity_name"] for c in t.index]
    t["시장"] = [INFO[c]["market"] for c in t.index]
    t["합계"] = t["관련 문서"] + t["언급만 된 문서"]
    return t.sort_values(["관련 문서", "합계"], ascending=False)[["종목", "시장", "관련 문서", "언급만 된 문서", "합계"]]

## 3. 예시 ① 경제뉴스 '계약/수주' 기사 × 중앙경제지 — 상장사만
앱의 이슈 프리셋과 언론사 묶음을 그대로 씁니다. 300건(3페이지)까지 받습니다.

In [6]:
q1 = f"({ISSUE_PRESETS['계약/수주']}) and {PUBLISHER_GROUPS['중앙경제지']}"
news = match_listed("news", ["economy"], q1, pages=3)
stocks = by_stock(news)
print("\n시장별 종목 수(관련+언급):", stocks.시장.value_counts().to_dict())
stocks.head(15)

[news ['economy']] '((수주 and 체결) or (수주 and 공급) or (계약 and 체결) or (계약 ' 총 584건 중 300건 받음 → 관련종목에 상장사 133건, 언급까지 포함하면 213건 (+80)

시장별 종목 수(관련+언급): {'KOSPI': 128, 'KOSDAQ': 67, 'KONEX': 1}


,종목,시장,관련 문서,언급만 된 문서,합계
005930,삼성전자,KOSPI,8,43,51
047050,포스코인터내셔널,KOSPI,6,2,8
307180,아이엘,KOSDAQ,5,1,6
290650,엘앤씨바이오,KOSDAQ,5,1,6
009410,태영건설,KOSPI,4,0,4
066570,LG전자,KOSPI,3,4,7
064350,현대로템,KOSPI,3,2,5
035720,카카오,KOSPI,3,2,5
005490,POSCO홀딩스,KOSPI,3,1,4
006360,GS건설,KOSPI,3,1,4


In [7]:
news[["created_at", "publisher", "title", "관련종목", "언급종목"]].head(12)

,created_at,publisher,title,관련종목,언급종목
0,2026-11-24T21:06,이데일리,최대 17억 시세차익에…잠래아 '줍줍' 5.3만명 몰려,[삼성물산],[]
1,2026-11-24T19:00,이데일리,급한 불 껐지만…지배구조·당국 조사·수익성 검증 과제 산적,[],"[SK하이닉스, 삼성전자, SK텔레콤, SK, NAVER]"
2,2026-11-24T18:44,디지털타임스,"포스코그룹, 美 셰일가스전 품었다… 장인화 ‘트리플 코어’ 본궤도",[포스코인터내셔널],[]
3,2026-11-24T18:42,디지털타임스,"중국 유탄 또 맞았다… K-배터리, 콩고發 공급망 리스크",[],"[삼성SDI, SK, LG에너지솔루션]"
4,2026-11-24T18:28,파이낸셜뉴스,"무보, 방산 중소기업 해외시장 진출 지원",[],[한화에어로스페이스]
5,2026-11-24T18:27,파이낸셜뉴스,"삼성전자, 승진 문턱 높았다… HBM4 주역엔 확실한 보상",[삼성전자],"[삼성SDI, 삼성전기]"
6,2026-11-24T18:19,파이낸셜뉴스,"""반도체·AI 투자심리 살아났지만 수익성 우려 여전""",[],"[SK하이닉스, 대신증권, 삼성전자, 키움증권]"
7,2026-11-24T18:18,이데일리,"""금리는 지켰는데 배당은?""…인천 스퀘어원, 홈플러스 빈자리 채우기 '숙제'",[서부T&D],[신한서부티엔디리츠]
8,2026-11-24T18:18,파이낸셜뉴스,'AI 동맹' 삼성·SK 한숨 돌렸다 [젠슨 황 복귀],[],"[SK하이닉스, 삼성전자, SK]"
9,2026-11-24T18:18,한국경제,"美 셰일가스전 품은 포스코인터 ""북미 생산거점 확보""",[포스코인터내셔널],[]


## 4. 예시 ② 증권사 리포트 '반도체' — 업종·시황 리포트도 상장사 언급으로 잡힌다
기업 리포트는 `securities`에 종목이 붙지만, 시황·산업 리포트는 대개 비어 있고 본문 언급(`named_entities`)에만 회사가 있습니다.

In [8]:
rep = match_listed("research", [], "반도체", pages=2)
by_stock(rep).head(12)

[research []] '반도체' 총 130건 중 130건 받음 → 관련종목에 상장사 27건, 언급까지 포함하면 95건 (+68)


,종목,시장,관련 문서,언급만 된 문서,합계
016360,삼성증권,KOSPI,5,8,13
006800,미래에셋증권,KOSPI,3,0,3
217500,러셀,KOSDAQ,2,1,3
005930,삼성전자,KOSPI,1,40,41
000660,SK하이닉스,KOSPI,1,29,30
034730,SK,KOSPI,1,13,14
005490,POSCO홀딩스,KOSPI,1,5,6
017670,SK텔레콤,KOSPI,1,4,5
052710,아모텍,KOSDAQ,1,1,2
028050,삼성E&A,KOSPI,1,1,2


## 5. 예시 ③ 공시 '단일판매·공급계약' — 시장별 건수
공시는 거의 모두 `securities`에 회사가 붙어 있습니다.

In [9]:
disc = match_listed("company", ["disclosure"], "title:단일판매", pages=2)
s3 = by_stock(disc, include_mentions=False)
print("시장별 공시 건수:", disc._rel.explode().map(lambda c: INFO.get(c, {}).get("market")).value_counts().to_dict())
s3.head(10)

[company ['disclosure']] 'title:단일판매' 총 62건 중 62건 받음 → 관련종목에 상장사 62건, 언급까지 포함하면 62건 (+0)
시장별 공시 건수: {'KOSDAQ': 38, 'KOSPI': 24}


,종목,시장,관련 문서,언급만 된 문서,합계
347700,스피어,KOSDAQ,5,0,5
002990,금호건설,KOSPI,2,0,2
006260,LS,KOSPI,2,0,2
452430,사피엔반도체,KOSDAQ,2,0,2
009540,HD한국조선해양,KOSPI,1,1,2
294870,IPARK현대산업개발,KOSPI,1,1,2
038870,에코심플렉스,KOSDAQ,1,0,1
013580,계룡건설,KOSPI,1,0,1
010960,삼호개발,KOSPI,1,0,1
166480,코아스템켐온,KOSDAQ,1,0,1


## 6. 예시 ④ 특허 '배터리'(2025년) — 출원인 중 상장사
특허의 `securities.symbol`에는 종목코드 대신 **출원인 회사 이름**(예: '엘지에너지솔루션', '현대자동차')이 들어 있어
이름만으로는 상장사 목록('LG에너지솔루션', '현대차')과 맞지 않습니다. 같은 항목에 있는 **법인번호·사업자번호**로 맞춥니다.
특허는 공개까지 시차가 있어 최근 몇 달 건수가 적습니다.

In [10]:
pat = match_listed("patent", [], "배터리", date_from="2025-01-01", date_to="2025-12-31", pages=3)
by_stock(pat).head(12)

[patent []] '배터리' 총 3,458건 중 300건 받음 → 관련종목에 상장사 104건, 언급까지 포함하면 104건 (+0)


,종목,시장,관련 문서,언급만 된 문서,합계
373220,LG에너지솔루션,KOSPI,60,0,60
015760,한국전력,KOSPI,7,0,7
051910,LG화학,KOSPI,7,0,7
005380,현대차,KOSPI,4,0,4
000270,기아,KOSPI,4,0,4
417010,나노팀,KOSDAQ,3,0,3
006400,삼성SDI,KOSPI,3,0,3
108670,LX하우시스,KOSPI,3,0,3
009830,한화솔루션,KOSPI,2,0,2
005930,삼성전자,KOSPI,2,0,2


In [11]:
# 이름 매칭만 했을 때와 비교 — 식별번호 매칭이 왜 필요한지
_, raw = fetch_docs("patent", [], "배터리", "2025-01-01", "2025-12-31", pages=1)
apps = [e for x in raw for e in (x.get("securities") or [])]
by_name = sum(1 for e in apps if e.get("name") in BY_NAME or str(e.get("symbol", "")).split(":")[-1] in INFO)
by_id = sum(1 for e in apps if to_code(e))
print(f"출원인 항목 {len(apps)}개 중 상장사로 판정: 이름·코드만 {by_name}개 → 식별번호까지 {by_id}개")

출원인 항목 80개 중 상장사로 판정: 이름·코드만 16개 → 식별번호까지 42개


## 7. 종목별 문서 수를 한 번에 — `DocumentAggregation`
문서를 받지 않고 종목별 건수만 필요하면 집계 한 번이면 됩니다. 위에서 받은 300건이 아니라 **검색 결과 전체**를 셉니다.
키는 대개 6자리 종목코드지만, 검색어·카테고리에 따라 DART 고유번호(8자리)·NICE 코드·회사 이름(특허)이 섞이므로
상장 종목코드인 것만 남깁니다.

In [12]:
c = ds(f'DocumentAggregation(["news"],["economy"],{json.dumps(q1, ensure_ascii=False)},"securities.symbol:200",'
       f'date_from={D_FROM},date_to={D_TO})')["data"]["pods"][1]["content"]
agg = pd.DataFrame(c["data"])[["key", "count"]]
agg["상장사"] = agg.key.isin(INFO.keys())
print(f"집계 키 {len(agg)}개 중 상장 종목코드 {agg.상장사.sum()}개, 그 밖(비상장·고유번호 등) {(~agg.상장사).sum()}개")
top = agg[agg.상장사].head(15).copy()
top["종목"] = top.key.map(lambda c: INFO[c]["entity_name"])
top["시장"] = top.key.map(lambda c: INFO[c]["market"])
top[["종목", "시장", "count"]].rename(columns={"count": "관련 기사 수"}).reset_index(drop=True)

집계 키 139개 중 상장 종목코드 139개, 그 밖(비상장·고유번호 등) 0개


,종목,시장,관련 기사 수
0,삼성전자,KOSPI,17
1,HMM,KOSPI,10
2,GS리테일,KOSPI,9
3,GS건설,KOSPI,7
4,삼성물산,KOSPI,7
5,포스코인터내셔널,KOSPI,6
6,엘앤씨바이오,KOSDAQ,6
7,SK하이닉스,KOSPI,5
8,태영건설,KOSPI,5
9,현대엘리베이터,KOSPI,5


## 8. 받은 문서를 이슈 카테고리·관심종목으로 거르기
검색은 넓게 하고, 받은 뒤 **제목**에 이슈 키워드 묶음이 맞는 문서만 남깁니다(`a and b`는 둘 다, `or`는 하나라도).
관심종목 필터는 관련·언급 종목 중 하나라도 관심종목이면 남깁니다.

In [13]:
def issue_match(title, preset):
    """ISSUE_PRESETS 식('(a and b) or c')을 제목에 적용 → 맞은 항목 목록."""
    hits = []
    for term in preset.split(" or "):
        words = [w.strip(" ()") for w in term.split(" and ")]
        if all(w and w in title for w in words):
            hits.append("+".join(words))
    return hits

wide = match_listed("news", ["economy"], "*", pages=3)        # 같은 기간 경제뉴스 300건, 상장사 문서만
counts = {}
for name, preset in ISSUE_PRESETS.items():
    counts[name] = int(wide.title.map(lambda t: bool(issue_match(t, preset))).sum())
print(f"상장사 경제뉴스 {len(wide)}건 중 이슈별 제목 매칭:", counts)
wide["이슈"] = wide.title.map(lambda t: [n for n, p in ISSUE_PRESETS.items() if issue_match(t, p)])
wide[wide.이슈.str.len() > 0][["created_at", "title", "이슈", "관련종목"]].head(8)

[news ['economy']] '*' 총 21,189건 중 300건 받음 → 관련종목에 상장사 50건, 언급까지 포함하면 109건 (+59)
상장사 경제뉴스 109건 중 이슈별 제목 매칭: {'계약/수주': 0, '인수/투자': 8, '실적': 0, '증자/감자': 0, '회계/감사': 0, '소송/부도/회생': 0}


,created_at,title,이슈,관련종목
9,2026-11-24T22:41,"‘드디어, 마침내!’ 외국인 투자 돌아오나…‘이때’가 진짜 분수령[투자360]",[인수/투자],[]
29,2026-11-24T21:41,저PBR 잡겠다 공언했는데…코스피 PBR 결국 작년으로 회귀 [투자360],[인수/투자],[]
31,2026-11-24T21:33,"[삼성바이오 인사]① 분할 1년, 법인 관리 보강·본업 전문가 중용",[인수/투자],[]
37,2026-11-24T21:10,다올투자證 경영권 매각설 수면 위로,[인수/투자],[]
42,2026-11-24T21:05,2배·3배 '간 큰' 고위험 투자…급등락에 손실↑,[인수/투자],[삼성전자]
50,2026-11-24T20:43,"‘트리플 코어’ 중 에너지 확보…포스코, 미 셰일가스전 인수 완료",[인수/투자],[POSCO홀딩스]
51,2026-11-24T20:41,"‘한번이면 됐지 두번 안 당해’ 앤트로픽 투자, 스페이스X보다 1/4 불과 [투자360]",[인수/투자],[삼성증권]
53,2026-11-24T20:36,"현대모비스, 램프사업 분할안 81.5% 찬성 가결…매수청구권 변수 남아",[인수/투자],[현대모비스]


In [14]:
watch = set(WATCHLIST)
mine = wide[wide.apply(lambda r: bool(watch & (set(r._rel) | set(r._men))), axis=1)]
print(f"관심종목 {[INFO[c]['entity_name'] for c in WATCHLIST if c in INFO]} 관련·언급 문서 {len(mine)}건")
mine[["created_at", "publisher", "title", "관련종목", "언급종목"]].head(8)

관심종목 ['삼성전자', 'SK하이닉스', '카카오', '펌텍코리아'] 관련·언급 문서 35건


,created_at,publisher,title,관련종목,언급종목
1,2026-11-24T23:28,스포츠경향,달콤한 보상 찾는 연말 맞아 ‘겨울 디저트 옷’ 입은 유통업계,[카카오],[]
2,2026-11-24T23:21,연합뉴스TV,삼성 인사도 반도체 독주…로봇에도 힘 실었다,[],"[삼성전자, 삼성전기, LG전자]"
4,2026-11-24T22:56,이데일리,"앤스로픽, 상장 4거래일 만에 공모가 아래로…엔비디아도 5%↓",[SK하이닉스],[]
8,2026-11-24T22:46,TV조선,"한국경쟁포럼 세종 워크숍 ""AI 에이전트가 수고 덜어주지만 새 게이트키퍼 우려""",[],"[NAVER, 카카오]"
13,2026-11-24T22:24,KBS,“첨단산업 유치 위해 전기요금 인하해야”,[삼성전자],[삼성에스디에스]
15,2026-11-24T22:03,경기일보,‘명픽’ 이광수의 ‘코스피 5000 붕괴 눈물’...그 영상 공유한 조국,[삼성전자],[]
16,2026-11-24T22:01,한국경제,"""400만닉스는 커녕 100만닉스도 위태""…폭락 맞춘 BNK의 경고 [종목+]",[SK하이닉스],[]
26,2026-11-24T21:45,경향신문,젠슨 황 복귀에 ‘메모리 깐부’ 삼전닉스 안도…미 ‘대중 수출통제’ 여전히 변수,[삼성전자],"[SK하이닉스, SK]"


## 결과 읽는 법과 한계
- **관련 종목(`securities`)은 문서의 주제 회사, 언급 종목(`named_entities`)은 본문에 나온 회사**입니다. 언급까지 넣으면
  뉴스에서 상장사 문서가 20~30% 더 잡히지만, 지나가듯 언급된 경우도 섞입니다.
- **동명 오인식**이 있습니다(예: 'DB', '미래산업', '엠비디'). 의심스러운 종목은 제목으로 다시 확인하세요.
- **현재 상장 종목 기준**입니다. 그 사이 상장폐지된 회사는 상장사로 잡히지 않습니다.
- 특허는 공개 시차 때문에 최근 건수가 적고, 출원인 이름이 공식명이라 식별번호 매칭이 필요합니다.